# is-differentiable-flag — worked example 2: Closure makes is_differentiable sticky across multiple calls

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `is-differentiable-flag`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The `is_differentiable` flag is captured at wrap-time in a closure, making it permanently associated with the wrapped op. This is different from `grad_tracking_enabled`, which is read fresh on every call from the module-level global. Mixing the two shows that one flag is fixed-per-op while the other is live.

## Worked solution

Step 1: We create two wrapped ops: `wrapped_add` with `is_differentiable=True` and `wrapped_argmax` with `is_differentiable=False`. Each closure captures its flag independently.

Step 2: We build a tracked input tensor `x` (requires_grad=True).

Step 3: We call both wrapped ops and inspect outputs. `wrapped_add` produces a tracked output (Recipe attached), `wrapped_argmax` never does.

Step 4: We then flip `grad_tracking_enabled = False`. Now even `wrapped_add` produces untracked output (the global gate short-circuits). `wrapped_argmax` is unaffected — it was already producing untracked output.

Step 5: We flip back to `grad_tracking_enabled = True`. `wrapped_add` recovers; `wrapped_argmax` stays non-differentiable. This shows closure-stickiness vs global-freshness.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn, is_differentiable=True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        global grad_tracking_enabled
        requires_grad = (
            grad_tracking_enabled
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

wrapped_add = wrap_forward_fn(np.add, is_differentiable=True)
wrapped_argmax = wrap_forward_fn(np.argmax, is_differentiable=False)

x = MiniTensor(np.array([3.0, 1.0, 2.0]), requires_grad=True)

# Both called with tracking ON
add_out = wrapped_add(x, x)
argmax_out = wrapped_argmax(x)
print('add requires_grad (tracking on):', add_out.requires_grad)       # True
print('argmax requires_grad (tracking on):', argmax_out.requires_grad)  # False

# Flip global OFF: add loses tracking, argmax unchanged
grad_tracking_enabled = False
add_out2 = wrapped_add(x, x)
argmax_out2 = wrapped_argmax(x)
print('add requires_grad (tracking off):', add_out2.requires_grad)      # False
print('argmax requires_grad (tracking off):', argmax_out2.requires_grad) # False

# Flip global back ON: add recovers, argmax stays non-diff
grad_tracking_enabled = True
add_out3 = wrapped_add(x, x)
print('add requires_grad (tracking on again):', add_out3.requires_grad)  # True
print('add has recipe:', add_out3.recipe is not None)                     # True